In [ ]:
from pathlib import Path
import yaml
import numpy as np
import matplotlib.pyplot as plt

import torch
import fabio

from pyFAI.geometry import Geometry
from pyFAI.integrator.azimuthal import AzimuthalIntegrator

from maxima_vit.utils import (
    load_model,
    image_to_tensor,
    get_calibrant,
    get_detector,
    PeakOptimizer,
)

In [ ]:
CONFIG_PATH = Path()
MODEL_PATH = Path()
SCAN_PATH = Path()

CALIBRANT_NAME = "alpha_Al2O3"
DETECTOR_NAME = "Eiger2Cdte_1M"
WAVELENGTH_M = 0.5121261413149675e-10

In [ ]:
with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

calibrant_name = config.get("calibrant", CALIBRANT_NAME)
detector_name = config.get("detector", DETECTOR_NAME)
wavelength_m = config.get("wavelength", WAVELENGTH_M)
image_size = int(config["model"].get("image_size", 1056))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = load_model(str(MODEL_PATH), config).to(device)
model.eval()

scan = fabio.open(str(SCAN_PATH)).data.astype(np.float32)

x = image_to_tensor(scan, image_size).unsqueeze(0).to(device)
with torch.no_grad():
    pred = model(x).detach().cpu().numpy().flatten()

param_names = ["dist", "poni1", "poni2", "rot1", "rot2", "rot3"]
pred_dict = {k: float(v) for k, v in zip(param_names, pred)}
print("Predicted geometry:")
for k, v in pred_dict.items():
    print(f"  {k:>5}: {v:.8f}")

calibrant = get_calibrant(calibrant_name, wavelength_m)
detector = get_detector(detector_name)

geometry = Geometry(
    dist=pred_dict["dist"],
    poni1=pred_dict["poni1"],
    poni2=pred_dict["poni2"],
    rot1=pred_dict["rot1"],
    rot2=pred_dict["rot2"],
    rot3=pred_dict["rot3"],
    wavelength=calibrant.wavelength,
    detector=detector,
)

ai = AzimuthalIntegrator(
    dist=geometry.dist,
    poni1=geometry.poni1,
    poni2=geometry.poni2,
    rot1=geometry.rot1,
    rot2=geometry.rot2,
    rot3=geometry.rot3,
    detector=geometry.detector,
    wavelength=geometry.wavelength,
)

In [ ]:
ring_tth = np.asarray(calibrant.get_2th(), dtype=np.float32)
ring_tth = ring_tth[np.isfinite(ring_tth)]
ring_tth = ring_tth[ring_tth > 0]

if ring_tth.size == 0:
    raise ValueError("Calibrant has no valid ring positions for the selected wavelength.")

tth_max = float(ring_tth.max() * 1.05)
tth_axis = np.linspace(0.0, tth_max, 6000, dtype=np.float32)
ring_profile = np.zeros_like(tth_axis)

for tth in ring_tth:
    idx = int(np.argmin(np.abs(tth_axis - tth)))
    ring_profile[max(0, idx - 1): idx + 2] = 1.0

ring_map = ai.calcfrom1d(
    tth_axis,
    ring_profile,
    shape=scan.shape,
    dim1_unit="2th_rad",
    correctSolidAngle=False,
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(np.log1p(scan), cmap="magma", origin="lower")
ax.contour(ring_map, levels=[0.4], colors="cyan", linewidths=0.8)
ax.set_title("Predicted Geometry Overlay (cyan = pyFAI ring positions)")
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
optimizer = PeakOptimizer(
    image=scan,
    initial_geometry=geometry,
    calibrant=calibrant,
    exclude_border=300,
)

refiner = optimizer.optimize(initial_guess=[5, 0.1, 1.0])
if refiner is None:
    raise RuntimeError("Refinement did not converge to a valid solution.")

refined_geometry = Geometry(
    dist=float(refiner.dist),
    poni1=float(refiner.poni1),
    poni2=float(refiner.poni2),
    rot1=float(refiner.rot1),
    rot2=float(refiner.rot2),
    rot3=float(refiner.rot3),
    wavelength=calibrant.wavelength,
    detector=detector,
)

ai_refined = AzimuthalIntegrator(
    dist=refined_geometry.dist,
    poni1=refined_geometry.poni1,
    poni2=refined_geometry.poni2,
    rot1=refined_geometry.rot1,
    rot2=refined_geometry.rot2,
    rot3=refined_geometry.rot3,
    detector=refined_geometry.detector,
    wavelength=refined_geometry.wavelength,
)

refined_map = ai_refined.calcfrom1d(
    tth_axis,
    ring_profile,
    shape=scan.shape,
    dim1_unit="2th_rad",
    correctSolidAngle=False,
)

pred_vec = np.array([
    geometry.dist,
    geometry.poni1,
    geometry.poni2,
    geometry.rot1,
    geometry.rot2,
    geometry.rot3,
], dtype=np.float64)

refined_vec = np.array([
    refined_geometry.dist,
    refined_geometry.poni1,
    refined_geometry.poni2,
    refined_geometry.rot1,
    refined_geometry.rot2,
    refined_geometry.rot3,
], dtype=np.float64)

delta_vec = refined_vec - pred_vec

print("Refined geometry:")
for name, value in zip(param_names, refined_vec):
    print(f"  {name:>5}: {value:.8f}")

print("\nDelta (refined - predicted):")
for name, value in zip(param_names, delta_vec):
    print(f"  {name:>5}: {value:+.3e}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6), constrained_layout=True)

axes[0].imshow(np.log1p(scan), cmap="magma", origin="lower")
axes[0].contour(ring_map, levels=[0.4], colors="cyan", linewidths=0.8)
axes[0].set_title("Predicted geometry")
axes[0].set_axis_off()

axes[1].imshow(np.log1p(scan), cmap="magma", origin="lower")
axes[1].contour(refined_map, levels=[0.4], colors="lime", linewidths=0.8)
axes[1].set_title("Refined geometry")
axes[1].set_axis_off()

plt.show()